## Imports

In [1]:
import os
import pandas as pd
import numpy as np

from tqdm import tqdm
import gc

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error

from materials_stock_libs.utils import (
    generate_future_dates,
    normalizar_codigos,
    make_synthetic_data
    )

from materials_stock_libs.models.prophet import ProphetModels, LSTMResidualModel, plot_prophet
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 100)

import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

## Importing environment variables

In [2]:
ROOT_PATH = os.path.join(os.getcwd(), '..')
DATA_PATH = os.path.join(ROOT_PATH, 'data')
OUTPUTS_PATH = os.path.join(ROOT_PATH, 'outputs')
MODEL_PATH = os.path.join(ROOT_PATH, 'trained_models')

DATA_CEN_01 = os.path.join(DATA_PATH, 'DS_CEN_01_EAN')
DATA_PATH_CEN_02 = os.path.join(DATA_PATH, 'DS_CEN_02_EAN')

np.random.seed(42)

## Data Loading

## Model Training

### Model 01 - Prophet with holdout

In [9]:
df_ext_pivot_prod_01 = df_ext_pivot[df_ext_pivot['PRODUCT_DESC'] == 'LOSARTAN POTAS.MG (N.Q)']
df_ext_pivot_prod_01 = df_ext_pivot_prod_01[df_ext_pivot_prod_01['UF'] == 'CE'].copy()
pm = ProphetModels(date_col='DT_EMISSAO', target_col='UNIDADES', seed=42, model_dir='meu_pipeline_de_forecast')

#### 1. Preprocessing - temporal split

In [13]:
# PARA HOLDOUT
splits = pm.time_train_test_split_monthly(df_ext_pivot_prod_01, test_months=0.2)
train_df, test_df = splits[0]

print(f"Tamanho do treino: {len(train_df)}, Tamanho do teste: {len(test_df)}")

train_df = train_df.rename(columns={'DT_EMISSAO': 'ds', 'QUANTIDADE': 'y'})
test_df = test_df.rename(columns={'DT_EMISSAO': 'ds', 'QUANTIDADE': 'y'})

# PARA WALK-FORWARD
# splits_wf = pm.time_train_test_split_monthly(df_ext_pivot_prod_01, test_months=0.2, walk_forward=True, n_splits=5)

# print(f"Total de splits gerados: {len(splits_wf)}")

# # Itere sobre a lista
# for i, (train_split, test_split) in enumerate(splits_wf):
#     print(f"Split {i+1}: Treino={len(train_split)}, Teste={len(test_split)}")

# for i in range(len(splits_wf)):
#     splits_wf[i] = (
#         splits_wf[i][0].rename(columns={'DT_EMISSAO': 'ds', 'QUANTIDADE': 'y'}),
#         splits_wf[i][1].rename(columns={'DT_EMISSAO': 'ds', 'QUANTIDADE': 'y'})
#     )

Tamanho do treino: 38, Tamanho do teste: 10


#### 2. Hyperparameter Optimization

In [9]:
best_params, _ = pm.optimize_prophet_hyperparams(pd.concat([train_df, test_df], axis=0).reset_index(drop=True), n_trials=10, horizon_months=3)
prophet_model = pm.train_prophet(pd.concat([train_df, test_df], axis=0).reset_index(drop=True), **best_params)

[I 2025-11-04 23:41:38,798] A new study created in memory with name: no-name-3a23eea7-6273-4f65-b83d-69fa56ad4192
Optuna Prophet Optimization:   0%|          | 0/10 [00:00<?, ?it/s]23:41:38 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] done processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] start processing
23:41:39 - cmdstanpy - INFO - Chain [1] done processing
23:41:39 - cmdstanpy - INFO - Chain [1] done processing
23:41:39 - cmdstanpy - INFO - Chain [1] done processing
23:41:39 - cmdstanpy - INFO - Chain [1] done processing
23:41:39 - cmdstanpy - INF


🏆 Melhor RMSE: 34.1477
Melhores parâmetros: {'changepoint_prior_scale': 0.0012522144737621792, 'seasonality_prior_scale': 0.21643473824872775, 'holidays_prior_scale': 8.257401560573657, 'seasonality_mode': 'multiplicative', 'changepoint_range': 0.9025919322387449}
✅ Melhor modelo Prophet treinado e salvo em 'self.prophet_model_'.
Treinando modelo Prophet...
✅ Modelo Prophet treinado e salvo em 'self.prophet_model_'.


#### Inference with pure prophet

In [10]:
df_full_prophet = pd.concat([train_df, test_df], ignore_index=True)

# Gera previsões Prophet para o dataset completo (para calcular resíduos)
forecast_full = pm.infer_prophet(df_full_prophet, horizon=3) # Usa pm.prophet_model_

Executando inferência Prophet (horizonte=3)...
✅ Previsão Prophet concluída.


### Model 02 - Prophet + LSTM with walk-forward

In [ ]:
categorical_cols = ['UF']#, , 'DISTRIBUIDOR',	'MEDICAMENTO',	'ATC1',	'NEC1',	'AREA_FARMACIA']
numerical_cols = ['residual', 'month_sin', 'month_cos'] # ,  'residual' is the main numerical feature

# Fit and save encoders in pm.encoders_
df_pre, _ = pm.preprocess_features(forecast_full, categorical_cols, fit_encoders=True)

# Fit and save scaler in pm.scaler_
df_pre, _ = pm.fit_scaler(df_pre, numerical_cols, scaler_class=MinMaxScaler, feature_range=(-1, 1))

# Calculate cardinalities and embeddings for LSTM
cat_cardinalities = [len(pm.encoders_[col].classes_) for col in categorical_cols]
embed_dims = [min(50, int(card ** 0.25) * 4) for card in cat_cardinalities]

Ajustando LabelEncoders...
✅ 0 encoders ajustados e salvos em 'self.encoders_'.
Ajustando Scaler...
✅ Scaler (MinMaxScaler) ajustado e salvo em 'self.scaler_'.


In [37]:
df_pre[['y', 'ds']].value_counts().reset_index()

print(df_pre['y'].min())
print(df_pre['y'].max())

278.0
471.0


In [ ]:
# Train the LSTM and save it in pm.lstm_model_
results_df, _ = pm.walk_forward_train(
    df=df_pre,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    cat_cardinalities=cat_cardinalities,
    embed_dims=embed_dims,
    window_size=6,
    horizon=3,
    lstm_epochs=5,
    incremental=True
)

✅ Modelo Prophet e encoders encontrados.
Treinando no device: cuda


Walk-Forward Training:   0%|          | 0/18 [00:00<?, ?it/s]00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] done processing
Walk-Forward Training:   6%|▌         | 1/18 [00:00<00:02,  7.05it/s]00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] done processing
Walk-Forward Training:  11%|█         | 2/18 [00:00<00:01,  8.40it/s]00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] done processing
Walk-Forward Training:  17%|█▋        | 3/18 [00:00<00:01,  8.21it/s]00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] done processing
Walk-Forward Training:  22%|██▏       | 4/18 [00:00<00:01,  7.75it/s]00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] done processing
00:05:55 - cmdstanpy - INFO - Chain [1] start processing
00:05:55 - cmdstanpy - INFO - Chain [1] d

✅ Gráficos por época salvos em outputs/epoch_panels/

✅ Walk-forward completo.
🏆 Melhor RMSE Global: 12.0576


In [ ]:
print("\nWalk-Forward Results:")
print(results_df)


Resultados do Walk-Forward:
    train_end test_start   test_end  best_rmse   best_mae
0  2023-12-01 2024-01-01 2024-03-01  16.100597  16.070691
1  2024-01-01 2024-02-01 2024-04-01  59.530431  58.871206
2  2024-02-01 2024-03-01 2024-05-01  55.497757  46.984811
3  2024-03-01 2024-04-01 2024-06-01  29.530123  23.617703
4  2024-04-01 2024-05-01 2024-07-01  22.670055  20.987689
5  2024-05-01 2024-06-01 2024-08-01  29.599984  24.035650
6  2024-06-01 2024-07-01 2024-09-01  25.023646  19.051651
7  2024-07-01 2024-08-01 2024-10-01  47.949718  44.500181
8  2024-08-01 2024-09-01 2024-11-01  50.405002  47.064125
9  2024-09-01 2024-10-01 2024-12-01  19.116056  18.987811
10 2024-10-01 2024-11-01 2025-01-01  24.981847  23.897745
11 2024-11-01 2024-12-01 2025-02-01  12.057625  10.662130
12 2024-12-01 2025-01-01 2025-03-01  23.345315  18.365019
13 2025-01-01 2025-02-01 2025-04-01  23.817755  23.807493
14 2025-02-01 2025-03-01 2025-05-01  17.631578  14.543398
15 2025-03-01 2025-04-01 2025-06-01  13.688

In [48]:
results_rmse = results_df['best_rmse']
mask_nao_infinito = ~np.isinf(results_rmse)
results_rmse_sem_inf = results_rmse[mask_nao_infinito]
results_rmse_sem_inf.mean()

29.434101626857334

In [16]:
# ----------------------------
# 5. Salvar o Pipeline Inteiro
# ----------------------------
# Não precisa passar argumentos, ele salva o estado (prophet, lstm, encoders, scaler)
pm.save_models()

💾 Salvando artefatos em meu_pipeline_de_forecast...
✅ Modelo Prophet salvo.
✅ Modelo LSTM salvo.
✅ 6 encoders salvos.
✅ Scaler salvo.
💾 Salvamento concluído (modo: HYBRID).


### Carga e inferência em produção

In [17]:
# ----------------------------
# 6. Carregar e Inferir (Exemplo de produção)
# ----------------------------
print("\n" + "="*30)
print("INICIANDO TESTE DE CARGA E INFERÊNCIA")
print("="*30 + "\n")

HORIZONTE_PREVISAO = 3

# Instancia um NOVO pipeline para simular um processo de produção
pm_prod = ProphetModels(
    date_col='DT_EMISSAO', 
    target_col='UNIDADES', 
    seed=42, 
    model_dir='meu_pipeline_de_forecast'
)

# Carrega o estado salvo (modelos, encoders, scaler) para dentro de pm_prod
pm_prod.load_models(
    model_class=LSTMResidualModel,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols
)


INICIANDO TESTE DE CARGA E INFERÊNCIA

🔄 Carregando artefatos de meu_pipeline_de_forecast...
✅ Modelo Prophet carregado.
✅ 6 LabelEncoders carregados.
✅ Scaler carregado.
✅ Modelo LSTM carregado (modo híbrido ativado).

Modo carregado: HYBRID


In [31]:
# Em produção, você teria apenas os dados históricos
df_pre_dropped = df_pre.drop(columns=['yhat', 'residual', 'month_sin', 'month_cos'])
df_historico_prod = df_pre_dropped.rename(columns={'DT_EMISSAO': 'ds', 'UNIDADES': 'y'})
print(df_historico_prod.head(3))
df_historico_prod.shape

             EAN     PRODUCT_DESC  UF  DISTRIBUIDOR  MEDICAMENTO  ATC1  NEC1  AREA_FARMACIA  \
0  7892950000000  BIO-VAGIN (A5U)   0             0            0     0     0              0   
1  7892950000000  BIO-VAGIN (A5U)   0             0            0     0     0              0   
2  7892950000000  BIO-VAGIN (A5U)   0             0            0     0     0              0   

          ds     RS_PC    RS_PPP     RS_PR      y  month  
0 2023-06-01  26053.65  20926.35  22542.30  405.0      6  
1 2023-07-01  28555.65  23562.75  24768.70  445.0      7  
2 2023-08-01  30214.65  24769.89  26215.86  471.0      8  


(27, 14)

In [32]:
forecast_prod = pm_prod.infer_prophet(df_historico_prod, horizon=HORIZONTE_PREVISAO)

Executando inferência Prophet (horizonte=3)...
✅ Previsão Prophet concluída.


In [1]:
# # 2. Inferência Híbrida (se o modo for 'hybrid')
# if pm_prod.pipeline_mode_ == "hybrid":
    
#     # O 'infer_hybrid' precisa das features categóricas para os meses futuros
#     # (Neste exemplo, vamos apenas repetir as últimas)
#     future_features = df_historico_prod[categorical_cols].iloc[-HORIZONTE_PREVISAO:].reset_index(drop=True)
    
#     # Adiciona as features categóricas ao 'forecast_prod'
#     # Pega as features do histórico
#     forecast_prod_com_features = forecast_prod.merge(
#         df_historico_prod[['ds'] + categorical_cols], on='ds', how='left'
#     )
    
#     # Preenche as features dos meses futuros (índices -HORIZONTE_PREVISAO até o fim)
#     for i, col in enumerate(categorical_cols):
#             forecast_prod_com_features.loc[
#                 forecast_prod_com_features.index[-HORIZONTE_PREVISAO:], col
#             ] = future_features[col].values

#     # Agora, chame a inferência híbrida (não precisa passar modelos/scalers)
#     preds_hybrid, preds_prophet, dates = pm_prod.infer_hybrid(
#         df=forecast_prod_com_features, # DF com dados hist + features futuras
#         forecast_all=forecast_prod_com_features, # DF com yhat + features futuras
#         categorical_cols=categorical_cols,
#         feature_cols=numerical_cols,
#         window=6,
#         horizon=HORIZONTE_PREVISAO
#     )
    
#     print("\nPrevisões Híbridas Geradas:")
#     print(pd.DataFrame({"data": dates, "previsao_hibrida": preds_hybrid, "previsao_prophet": preds_prophet}))

#     # Plotar
#     plot_prophet(forecast_prod, horizon=HORIZONTE_PREVISAO, preds_hybrid=preds_hybrid, train_df=train_df_prophet)

# else:
#     print("\nPrevisões Prophet (Puro) Geradas:")
#     print(forecast_prod.tail(HORIZONTE_PREVISAO))
#     plot_prophet(forecast_prod, horizon=HORIZONTE_PREVISAO)

In [ ]:
# TODO: IMPLEMENTAR AVALIAÇÃO DO MODELO USANDO SHAP + Prophet explainability:

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class ProphetModels:
    # ... (resto da classe)

    # ---------- 1) EXPLICAÇÃO SHAP DO LSTM RESIDUAL ----------
    def explain_lstm_shap(
        self,
        df_pre: pd.DataFrame,
        numerical_cols: list,
        categorical_cols: list,
        window_size: int = 6,
        sample_size: int = 256,
        top_k: int = 15,
        out_dir: str = "outputs",
        shap_html: str = "shap_beeswarm.html"
    ):
        """
        Calcula valores SHAP para o LSTM residual (que corrige o Prophet).
        Espera df_pre já preprocessado (categóricas label-encoded, numéricas escaladas).
        Retorna:
            - feature_importance_df: DataFrame com importâncias médias |SHAP| por feature
            - shap_values_np: array SHAP bruto (amostras x janela x features_concat)
            - X_concat_np: array de entrada concatenado usado no SHAP (amostras x janela x features_concat)
        """
        # --- Checagens de estado ---
        if getattr(self, "lstm_model_", None) is None:
            raise ValueError("self.lstm_model_ não encontrado. Treine o LSTM antes de explicar.")
        if getattr(self, "prophet_model_", None) is None:
            raise ValueError("self.prophet_model_ não encontrado. Treine o Prophet antes de explicar.")
        if "residual" not in numerical_cols:
            raise ValueError("numerical_cols deve conter 'residual' (o LSTM aprende a corrigir o Prophet).")
        req_cols = ["ds", "y"] + categorical_cols + numerical_cols
        faltam = [c for c in req_cols if c not in df_pre.columns]
        if faltam:
            raise ValueError(f"df_pre está sem colunas necessárias: {faltam}")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = self.lstm_model_.to(device).eval()

        # --- Monta tensores base (sem criar janelas ainda) ---
        X_num_full = torch.tensor(df_pre[numerical_cols].values, dtype=torch.float32, device=device)
        X_cat_full = torch.tensor(np.stack([df_pre[c].values for c in categorical_cols], axis=1),
                                  dtype=torch.long, device=device) if categorical_cols else torch.zeros(
                                    (len(df_pre), 0), dtype=torch.long, device=device)
        # y residual (em regra não precisamos dele pro SHAP, mas deixo aqui se quiser alinhar algo)
        # y_full = torch.tensor(df_pre["residual"].values, dtype=torch.float32, device=device).view(-1, 1)

        # --- Cria janelas (usa o método já da classe) ---
        X_num_seq, X_cat_seq, _ = self.create_sequences(X_num_full, X_cat_full, torch.zeros(len(df_pre), 1, device=device), window_size)
        if X_num_seq.shape[0] == 0:
            raise ValueError(f"Sem janelas suficientes para SHAP: n={len(df_pre)} <= window_size={window_size}")

        # Amostragem para o SHAP (para ser mais rápido)
        n_samples = X_num_seq.shape[0]
        idx = np.random.choice(n_samples, size=min(sample_size, n_samples), replace=False)
        Xn_sample = X_num_seq[idx]                        # (batch, window, n_num)
        Xc_sample = X_cat_seq[idx]                        # (batch, window, n_cat)

        # Concatena num + cat em uma única entrada (DeepExplainer exige 1 input)
        def _concat(Xn, Xc):
            return torch.cat([Xn, Xc.float()], dim=-1)

        X_concat = _concat(Xn_sample, Xc_sample)          # (batch, window, n_num+n_cat)
        n_num = Xn_sample.shape[-1]
        n_cat = Xc_sample.shape[-1] if categorical_cols else 0
        feature_names = numerical_cols + categorical_cols

        # Wrapper para o LSTM aceitar 1 tensor concatenado
        class LSTMWrapper(nn.Module):
            def __init__(self, base_model, n_num, n_cat):
                super().__init__()
                self.base = base_model
                self.n_num = n_num
                self.n_cat = n_cat
            def forward(self, x):
                # x: (batch, window, n_num+n_cat)
                x_num = x[:, :, :self.n_num]
                x_cat = x[:, :, self.n_num:].long() if self.n_cat > 0 else None
                return self.base(x_num, x_cat)

        wrapper = LSTMWrapper(model, n_num, n_cat).to(device).eval()

        # --- DeepExplainer ---
        # background = pequena amostra de fundo
        bg_size = min(64, X_concat.shape[0])
        background = X_concat[:bg_size].detach()

        # SHAP pode precisar de tensores no CPU; DeepExplainer aceita CUDA também.
        explainer = shap.DeepExplainer(wrapper, background)
        shap_values = explainer.shap_values(X_concat)  # retorna tensor/array com mesmo shape do input

        # Converte para numpy
        shap_values_np = shap_values if isinstance(shap_values, np.ndarray) else shap_values.detach().cpu().numpy()
        X_concat_np = X_concat.detach().cpu().numpy()

        # --- Agregação por feature (média de |SHAP| ao longo da janela e das amostras) ---
        # shap_values_np: (batch, window, n_features_concat)
        # agregamos em batch e window => (n_features_concat,)
        mean_abs = np.nanmean(np.abs(shap_values_np), axis=(0, 1))
        # Separa blocos num + cat
        fi_num = mean_abs[:n_num]
        fi_cat = mean_abs[n_num:n_num + n_cat] if n_cat > 0 else np.array([])

        feature_importance = np.concatenate([fi_num, fi_cat])
        feature_importance_df = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_shap": feature_importance
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

        # (Opcional) Beeswarm padrão do SHAP salvo em HTML
        os.makedirs(out_dir, exist_ok=True)
        try:
            shap.plots.beeswarm(shap.Explanation(
                values=shap_values_np.reshape(shap_values_np.shape[0], -1),
                # para beeswarm clássico com janela, achatamos a janela por conveniência
                data=X_concat_np.reshape(X_concat_np.shape[0], -1),
                feature_names=[f"{nm}(t-{window_size-k})" for k in range(window_size, 0, -1) for nm in feature_names]
            ), show=False)
            import matplotlib.pyplot as plt
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, shap_html.replace(".html", ".png")), dpi=150, bbox_inches="tight")
            plt.close()
        except Exception as _:
            # se o ambiente não tiver backend gráfico
            pass

        return feature_importance_df, shap_values_np, X_concat_np

    # ---------- 2) DASHBOARD PLOTLY HÍBRIDO ----------
    def dashboard_hibrido_plotly(
        self,
        df: pd.DataFrame,
        numerical_cols: list,
        categorical_cols: list,
        df_pre: pd.DataFrame,
        window_size: int,
        top_k: int = 10,
        out_html: str = "outputs/hybrid_dashboard.html"
    ):
        """
        Cria um dashboard Plotly com:
          (1) Série temporal: y (real) e yhat (Prophet) no período observado de df
          (2) Bar chart: top_k features por importância SHAP do LSTM residual
        Necessita do método explain_lstm_shap() rodando antes internamente.
        """
        # 1) Gerar yhat Prophet no histórico (sem futuro)
        df_ts = df[['ds', 'y']].dropna().copy().sort_values('ds')
        # Prophet precisa receber o DataFrame com coluna 'ds'
        in_sample = self.prophet_model_.predict(df_ts[['ds']].copy())
        df_ts = df_ts.merge(in_sample[['ds', 'yhat']], on='ds', how='left')

        # 2) SHAP no LSTM residual
        fi_df, shap_vals_np, _ = self.explain_lstm_shap(
            df_pre=df_pre,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            window_size=window_size,
            sample_size=256,
            top_k=top_k
        )
        fi_top = fi_df.head(top_k)

        # 3) Dashboard
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=False,
            vertical_spacing=0.12,
            subplot_titles=("Série temporal — y vs yhat (Prophet)", f"Importância de features (|SHAP| médio) — Top {top_k}")
        )

        # (1) Linha: y real
        fig.add_trace(
            go.Scatter(
                x=df_ts['ds'], y=df_ts['y'],
                mode='lines+markers', name='y (real)'
            ),
            row=1, col=1
        )
        # (1) Linha: yhat Prophet
        fig.add_trace(
            go.Scatter(
                x=df_ts['ds'], y=df_ts['yhat'],
                mode='lines+markers', name='yhat (Prophet)'
            ),
            row=1, col=1
        )

        # (2) Barras: SHAP
        fig.add_trace(
            go.Bar(
                x=fi_top['mean_abs_shap'][::-1],
                y=fi_top['feature'][::-1],
                orientation='h',
                name='|SHAP| médio'
            ),
            row=2, col=1
        )

        fig.update_layout(
            height=800,
            title_text="Dashboard Híbrido — Prophet + LSTM (explicabilidade)",
            showlegend=True
        )
        fig.update_yaxes(title_text="y", row=1, col=1)
        fig.update_xaxes(title_text="Data", row=1, col=1)
        fig.update_xaxes(title_text="Importância média (|SHAP|)", row=2, col=1)

        os.makedirs(os.path.dirname(out_html) or ".", exist_ok=True)
        fig.write_html(out_html)
        print(f"✅ Dashboard salvo em: {out_html}")

        return fi_df, fig

# USO:
# Depois do seu treino híbrido:
# self.prophet_model_, self.lstm_model_, self.encoders_, self.scaler_ prontos

categorical_cols = ['UF','DISTRIBUIDOR','MEDICAMENTO','ATC1','NEC1','AREA_FARMACIA']
numerical_cols = ['residual','month_sin','month_cos']  # ajuste conforme seu treino
window_size = 6

# df_pre: seu dataframe preprocessado (label-encoded + escalado) com ['ds','y', ...]
fi_df, shap_vals_np, X_concat_np = pm.explain_lstm_shap(
    df_pre=df_pre,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    window_size=window_size,
    sample_size=256,
    top_k=15
)

# Dashboard
pm.dashboard_hibrido_plotly(
    df=df[['ds','y']].rename(columns={'DT_EMISSAO':'ds','UNIDADES':'y'}) if 'DT_EMISSAO' in df.columns else df,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    df_pre=df_pre,
    window_size=window_size,
    top_k=10,
    out_html="outputs/hybrid_dashboard.html"
)
